# 3. Classification – Social Network Ads
**Dataset:** `Social_Network_Ads.csv` (400 rows) | **Task:** Predict whether a user `Purchased` the product  
**Algorithms (4):** Logistic Regression, K-Nearest Neighbours, Support Vector Machine, Random Forest

In [ ]:
import warnings; warnings.filterwarnings('ignore')
import numpy as np, pandas as pd
import matplotlib.pyplot as plt, seaborn as sns
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             confusion_matrix, roc_auc_score, roc_curve, classification_report)
sns.set_style('whitegrid')

## 1. Load & explore

In [ ]:
df = pd.read_csv('data/Social_Network_Ads.csv')
display(df.head()); print(df.shape); print(df.isnull().sum()); print(df['Purchased'].value_counts())
fig, ax = plt.subplots(1, 3, figsize=(16, 4))
sns.countplot(x='Purchased', data=df, ax=ax[0]); ax[0].set_title('Class balance')
sns.scatterplot(x='Age', y='EstimatedSalary', hue='Purchased', data=df, ax=ax[1]); ax[1].set_title('Age vs Salary')
sns.boxplot(x='Purchased', y='Age', data=df, ax=ax[2]); ax[2].set_title('Age by Purchased')
plt.tight_layout(); plt.show()

## 2. Preprocessing
`User ID` is an identifier (dropped). `Gender` is encoded. Scaling is placed **inside a pipeline** so it is fitted only on training data (no data leakage).

In [ ]:
X = pd.get_dummies(df.drop(['User ID', 'Purchased'], axis=1), drop_first=True)
y = df['Purchased']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=0, stratify=y)
print(X_train.shape, X_test.shape)

## 3. Train the 4 classifiers

In [ ]:
models = {
    'Logistic Regression': make_pipeline(StandardScaler(), LogisticRegression(random_state=0)),
    'KNN': make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=5)),
    'SVM (RBF)': make_pipeline(StandardScaler(), SVC(kernel='rbf', probability=True, random_state=0)),
    'Random Forest': RandomForestClassifier(n_estimators=200, random_state=0),
}
rows, fitted = [], {}
for name, m in models.items():
    m.fit(X_train, y_train); fitted[name] = m
    p = m.predict(X_test); prob = m.predict_proba(X_test)[:, 1]
    rows.append({'Model': name,
                 'Accuracy': accuracy_score(y_test, p), 'Precision': precision_score(y_test, p),
                 'Recall': recall_score(y_test, p), 'F1': f1_score(y_test, p),
                 'ROC-AUC': roc_auc_score(y_test, prob),
                 'CV Acc (5-fold)': cross_val_score(m, X, y, cv=5).mean()})
results = pd.DataFrame(rows).set_index('Model').round(4).sort_values('Accuracy', ascending=False)
display(results)

## 4. Confusion matrices & ROC curves

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(18, 4))
for ax, (name, m) in zip(axes, fitted.items()):
    sns.heatmap(confusion_matrix(y_test, m.predict(X_test)), annot=True, fmt='d', cmap='Blues', cbar=False, ax=ax)
    ax.set_title(name); ax.set_xlabel('Predicted'); ax.set_ylabel('Actual')
plt.tight_layout(); plt.show()

plt.figure(figsize=(6, 5))
for name, m in fitted.items():
    fpr, tpr, _ = roc_curve(y_test, m.predict_proba(X_test)[:, 1])
    plt.plot(fpr, tpr, label=f"{name} (AUC={roc_auc_score(y_test, m.predict_proba(X_test)[:, 1]):.3f})")
plt.plot([0, 1], [0, 1], 'k--'); plt.xlabel('FPR'); plt.ylabel('TPR'); plt.title('ROC curves'); plt.legend(); plt.show()

In [ ]:
best = results.index[0]
print('Best model:', best)
print(classification_report(y_test, fitted[best].predict(X_test)))

## 5. Conclusion
- All four classifiers were evaluated on the same stratified hold-out set with accuracy, precision, recall, F1, ROC-AUC and 5-fold CV.
- The non-linear models (SVM-RBF, KNN, Random Forest) beat Logistic Regression because the Age/Salary decision boundary is not linear.
- `Age` and `EstimatedSalary` are the main drivers of purchase behaviour (see the Age vs Salary scatter plot).